In [3]:
import numpy as np
import rasterio
import os
import re

In [4]:
def find_tif_files_with_date(directory):
    # Regular expression pattern to match files with YYYYMMDD and ending with .tif
    pattern = re.compile(r'^(\d{8}).*\.tif$')
    
    # Dictionary to store files grouped by their YYYYMMDD date
    files_by_date = {}

    # Loop through all files in the directory
    for filename in os.listdir(directory):
        match = pattern.match(filename)
        if match:
            date = match.group(1)  # Extract the YYYYMMDD part
            if date not in files_by_date:
                files_by_date[date] = []
            files_by_date[date].append(filename)

    return files_by_date


def udm_mask_PS_imagery(PS_raster_path, UDM2_mask_path):
    # Open the original raster image
    with rasterio.open(PS_raster_path) as src:
        # Read the raster data
        raster_data = src.read()
        raster_meta = src.meta

    # Open the mask raster
    with rasterio.open(UDM2_mask_path) as mask_src:
        # Read the mask data
        shadow_mask = mask_src.read(3).astype(bool) # hard-coded
        cloud_mask = mask_src.read(6).astype(bool) # hard-coded
        
    
    # Apply the mask: keep pixels where mask is non-zero (clear pixels), mask others (set to NaN)
    mask_data = shadow_mask + cloud_mask
    masked_data = np.where(mask_data == 0, raster_data, np.nan)

    # Create the output file name by appending the postfix to the original raster name
    output_path = PS_raster_path.replace('.tif', '_masked.tif')

    # Write the masked raster to a new file
    with rasterio.open(output_path, 'w', **raster_meta) as dst:
        dst.write(masked_data)

    return output_path


def mask_shadow_clouds(PS_image_directory):
    
    # Masking cloud and shadows
    AOI_PS_dir = PS_image_directory
    PS_file_dates = find_tif_files_with_date(AOI_PS_dir)

    for f_date in PS_file_dates:
        PS_files_list = PS_file_dates[f_date]
        for file_name in PS_files_list:
            if file_name.endswith('SR_clip.tif')|file_name.endswith('SR_8b_clip.tif')|\
            file_name.endswith('AnalyticMS_clip.tif')|file_name.endswith('AnalyticMS_SR_clip.tif'):
                PS_image_file = file_name
            elif file_name.endswith('udm2_clip.tif'):
                PS_mask_file = file_name

        PS_image_dir = os.path.join(AOI_PS_dir, PS_image_file)
        PS_mask_dir = os.path.join(AOI_PS_dir, PS_mask_file)

        _ = udm_mask_PS_imagery(PS_image_dir, PS_mask_dir)

## Masking shadows and clouds

In [5]:
# # PA
# # Gatesburg_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2019')
# # Gatesburg_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2020')
# # Gatesburg_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2021')
# # Gatesburg_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2022')
# #Gatesburg_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2023')
# # Gatesburg_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'PA', 'US-UC1_UC2', '2024')
# #US_HWB_2017_PS_dir = os.path.join(os.getcwd(), 'Data', 'US-HWB')

# # CA 
# # Bi1_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2018')
# # Bi1_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2019')
# # Bi1_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2020')
# # Bi1_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2021')
# # Bi1_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2022')
# #Bi1_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2023')
# #Bi1_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi1', '2024')

# # Bi2_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2018')
# # Bi2_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2019')
# # Bi2_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2020')
# # Bi2_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2021')
# # Bi2_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2022')
# Bi2_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2023')
# Bi2_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Bi2', '2024')

# Tw3_2017_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Tw3', '2017')
# Tw3_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'CA', 'US-Tw3', '2018')

# # IL
UiABC_2018_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2018')
UiABC_2019_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2019')
UiABC_2020_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2020')
UiABC_2021_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2021')
UiABC_2022_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2022')
UiABC_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2023')
UiABC_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'IL', 'US-UiABC', '2024')

# IN
VT12_2023_PS_dir = os.path.join(os.getcwd(), 'Data', 'IN', 'US-VT12', '2023')
VT12_2024_PS_dir = os.path.join(os.getcwd(), 'Data', 'IN', 'US-VT12', '2024')

In [6]:
# PA
# mask_shadow_clouds(Gatesburg_2023_PS_dir)

# CA
# mask_shadow_clouds(Bi1_2018_PS_dir)
# mask_shadow_clouds(Bi1_2019_PS_dir)
# mask_shadow_clouds(Bi1_2020_PS_dir)
# mask_shadow_clouds(Bi1_2021_PS_dir)
# mask_shadow_clouds(Bi1_2022_PS_dir)
# mask_shadow_clouds(Bi1_2023_PS_dir)
# mask_shadow_clouds(Bi1_2024_PS_dir)

# mask_shadow_clouds(Bi2_2018_PS_dir)
# mask_shadow_clouds(Bi2_2019_PS_dir)
# mask_shadow_clouds(Bi2_2020_PS_dir)
# mask_shadow_clouds(Bi2_2021_PS_dir)
# mask_shadow_clouds(Bi2_2022_PS_dir)
# mask_shadow_clouds(Bi2_2023_PS_dir)
# mask_shadow_clouds(Bi2_2024_PS_dir)

# mask_shadow_clouds(Tw3_2017_PS_dir)
# mask_shadow_clouds(Tw3_2018_PS_dir)

# IL
mask_shadow_clouds(UiABC_2018_PS_dir)
mask_shadow_clouds(UiABC_2019_PS_dir)
mask_shadow_clouds(UiABC_2020_PS_dir)
mask_shadow_clouds(UiABC_2021_PS_dir)
mask_shadow_clouds(UiABC_2022_PS_dir)
mask_shadow_clouds(UiABC_2023_PS_dir)
mask_shadow_clouds(UiABC_2024_PS_dir)

# IN
mask_shadow_clouds(VT12_2023_PS_dir)
mask_shadow_clouds(VT12_2024_PS_dir)


c:\Users\adadkhah\AppData\Local\miniconda3\envs\PA\Lib\site-packages\numpy\_core\_asarray.py:127: RuntimeWarning: invalid value encountered in cast
  arr = array(a, dtype=dtype, order=order, copy=None, subok=subok)


## For select dates

In [5]:
# PA
Gatesburg_PS_dir = os.path.join(os.getcwd(), 'For_select_dates', 'PS_data', 'GBF')


# CA 
Bi1_PS_dir = os.path.join(os.getcwd(), 'For_select_dates', 'PS_data', 'Bi1')

Bi2_PS_dir = os.path.join(os.getcwd(), 'For_select_dates', 'PS_data', 'Bi2')

Tw3_PS_dir = os.path.join(os.getcwd(), 'For_select_dates', 'PS_data', 'Tw3')

# IL
UiABC_PS_dir = os.path.join(os.getcwd(), 'For_select_dates', 'PS_data', 'UiABC')

# IN
VT12_PS_dir = os.path.join(os.getcwd(), 'For_select_dates', 'PS_data', 'VT12')


In [6]:
mask_shadow_clouds(Gatesburg_PS_dir)
mask_shadow_clouds(Bi1_PS_dir)
mask_shadow_clouds(Bi2_PS_dir)
mask_shadow_clouds(Tw3_PS_dir)
mask_shadow_clouds(UiABC_PS_dir)
mask_shadow_clouds(VT12_PS_dir)

c:\Users\adadkhah\AppData\Local\miniconda3\envs\PA\Lib\site-packages\numpy\_core\_asarray.py:127: RuntimeWarning: invalid value encountered in cast
  arr = array(a, dtype=dtype, order=order, copy=None, subok=subok)
